# Summary of Findings & Evidence-to-Decision — Runnable Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anacodicai/learn-notebooks/blob/main/notebooks/evidence-synthesis/15-2/summary-of-findings-and-etd.ipynb) &nbsp; [View on GitHub](https://github.com/anacodicai/learn-notebooks/blob/main/notebooks/evidence-synthesis/15-2/summary-of-findings-and-etd.ipynb)

> Companion to the **Summary of Findings & Evidence-to-Decision** article. Two artifacts turn
> pooled numbers into a decision: a **Summary of Findings (SoF)** table — one row per critical
> outcome with relative *and* absolute effects — and the **Evidence-to-Decision (EtD)** rule that
> outputs a **direction** (for/against) and a **strength** (**STRONG** vs **CONDITIONAL**). Watch
> a dramatic relative effect collapse into a *conditional* recommendation once absolute harm and
> variable values are on the table.

## What you'll build

1. `sof_row()` — the arithmetic that turns a relative effect + baseline risk into an **absolute** effect (per 1000).
2. `recommendation_strength()` — the fixed rule gating **STRONG** behind certainty **and** a clear balance **and** consistent values.
3. The article's worked example: a 2-row SoF (benefit + harm), a **CONDITIONAL FOR** vs a **STRONG FOR**, and a **Delphi** vote with a **discordance** check.

**No API key needed.** Pure Python (only `dataclasses`) — the panel supplies the judgments, the code guarantees the numbers and the mapping are reproducible.

### Setup (nothing to install)

In [1]:
from dataclasses import dataclass
print("ready — pure Python, no API key, no installs")

ready — pure Python, no API key, no installs


## Building a Summary-of-Findings row

The load-bearing arithmetic: the **absolute** effect is `baseline x RR - baseline` (fewer or more
events per 1000). The relative number is for comparability; the absolute number is what tells
you whether it *matters* for a real patient.

In [2]:
@dataclass
class SoFRow:
    outcome: str
    n_participants: int
    n_studies: int
    rr: float                  # relative risk (or OR) point estimate
    ci_low: float
    ci_high: float
    baseline_per_1000: float   # control-group risk per 1000
    certainty: str             # HIGH / MODERATE / LOW / VERY LOW


def sof_row(row: SoFRow) -> dict:
    """Turn a relative effect + baseline risk into the absolute effect (per 1000)."""
    with_treat = row.baseline_per_1000 * row.rr
    diff = with_treat - row.baseline_per_1000          # + = more events, - = fewer
    # propagate the CI to the absolute scale (same baseline)
    abs_low = row.baseline_per_1000 * row.ci_low - row.baseline_per_1000
    abs_high = row.baseline_per_1000 * row.ci_high - row.baseline_per_1000
    direction = "fewer" if diff < 0 else "more"
    return {
        "outcome": row.outcome,
        "participants_studies": f"{row.n_participants} ({row.n_studies} studies)",
        "certainty": row.certainty,
        "relative": f"RR {row.rr:.2f} ({row.ci_low:.2f}-{row.ci_high:.2f})",
        "absolute": (f"{row.baseline_per_1000:.0f} -> {with_treat:.0f} per 1000 "
                     f"= {abs(diff):.0f} {direction} "
                     f"(95% CI {abs(abs_low):.0f} to {abs(abs_high):.0f})"),
        "abs_diff_per_1000": round(diff, 1),
    }

print("sof_row() ready — absolute = baseline x RR - baseline")

sof_row() ready — absolute = baseline x RR - baseline


## The two SoF rows — a benefit and a harm

A guideline panel evaluates a drug for preventing a serious complication. The **benefit** is
HIGH-certainty (`RR 0.75` -> **20 fewer** per 1000); the **harm** is MODERATE (`RR 1.20`, CI
crosses 1.0 -> **6 more** per 1000). Both effects are printed on the absolute scale that drives
the decision.

In [3]:
benefit = sof_row(SoFRow("Serious complication", 4200, 6, 0.75, 0.66, 0.85, 80, "HIGH"))
harm    = sof_row(SoFRow("Serious adverse event", 3800, 5, 1.20, 0.95, 1.52, 30, "MODERATE"))

for r in (benefit, harm):
    print(f"{r['outcome']:<22} | {r['participants_studies']:<16} | {r['certainty']:<9} | {r['relative']}")
    print(f"{'':<22} | absolute: {r['absolute']}")
    print()
print("benefit:", benefit["absolute"])   # 80 -> 60 per 1000 = 20 fewer ...
print("harm   :", harm["absolute"])       # 30 -> 36 per 1000 = 6 more ...

Serious complication   | 4200 (6 studies) | HIGH      | RR 0.75 (0.66-0.85)
                       | absolute: 80 -> 60 per 1000 = 20 fewer (95% CI 27 to 12)

Serious adverse event  | 3800 (5 studies) | MODERATE  | RR 1.20 (0.95-1.52)
                       | absolute: 30 -> 36 per 1000 = 6 more (95% CI 2 to 16)

benefit: 80 -> 60 per 1000 = 20 fewer (95% CI 27 to 12)
harm   : 30 -> 36 per 1000 = 6 more (95% CI 2 to 16)


## Relative vs absolute — why the same ratio can mean 100x

The most-tested idea in the topic: a relative effect (`RR 0.50`) is stable across populations
but hides baseline risk. In a high-risk group it saves **100 per 1000**; in a low-risk group,
just **1 per 1000**. Same ratio, 100x difference in what a patient actually gains — which is why
you always report both.

In [4]:
for label, baseline in [("high-risk", 200), ("low-risk", 2)]:
    r = sof_row(SoFRow("Event", 1000, 1, 0.50, 0.40, 0.60, baseline, "HIGH"))
    print(f"{label:<10} RR 0.50 -> {r['absolute']}")

print("\nSame RR 0.50, 100x difference in ABSOLUTE benefit -> always report both.")

high-risk  RR 0.50 -> 200 -> 100 per 1000 = 100 fewer (95% CI 120 to 80)
low-risk   RR 0.50 -> 2 -> 1 per 1000 = 1 fewer (95% CI 1 to 1)

Same RR 0.50, 100x difference in ABSOLUTE benefit -> always report both.


## The strength rule — STRONG needs all three

A single strong result is not automatically a strong recommendation. `recommendation_strength()`
gates **STRONG** behind *all three*: solid certainty (HIGH/MODERATE) **and** a clear (non-close)
balance **and** consistent values. Any soft input deterministically downgrades to **CONDITIONAL**,
with the reason recorded.

In [5]:
_STRONG_CERTAINTY = {"HIGH", "MODERATE"}

def recommendation_strength(certainty: str, net_benefit: str, values_alignment: str) -> dict:
    """Fixed rule: STRONG needs solid certainty AND a clear balance AND consistent values.

    certainty        : HIGH / MODERATE / LOW / VERY LOW
    net_benefit      : 'clear_benefit' / 'close' / 'clear_harm'
    values_alignment : 'consistent' / 'variable'
    """
    if net_benefit == "clear_benefit":
        direction = "FOR"
    elif net_benefit == "clear_harm":
        direction = "AGAINST"
    else:                                   # 'close' -> lean but never strong
        direction = "CONDITIONAL-either-way"

    strong = (certainty in _STRONG_CERTAINTY
              and net_benefit in ("clear_benefit", "clear_harm")
              and values_alignment == "consistent")
    strength = "STRONG" if strong else "CONDITIONAL"

    reasons = []
    if certainty not in _STRONG_CERTAINTY:
        reasons.append("low/very-low certainty")
    if net_benefit == "close":
        reasons.append("close benefit-harm trade-off")
    if values_alignment == "variable":
        reasons.append("variable patient values")
    return {
        "direction": direction,
        "strength": strength,
        "verb": "recommend" if strength == "STRONG" else "suggest",
        "downgraded_because": reasons or ["none (strong criteria met)"],
    }

print("recommendation_strength() ready — STRONG is gated behind all three conditions")

recommendation_strength() ready — STRONG is gated behind all three conditions


## Walk the EtD — a real benefit, but a CONDITIONAL call

The benefit is HIGH certainty, but patients differ on how much they fear the adverse event
(**variable values**) and the harm CI crosses 1.0 (**uncertain harm**). Even a high-certainty
benefit yields only a **CONDITIONAL FOR** — "we *suggest*" — because STRONG needs certainty
*and* a clear balance *and* consistent values.

In [6]:
rec = recommendation_strength(certainty="HIGH",
                              net_benefit="clear_benefit",
                              values_alignment="variable")
print(rec["strength"], rec["direction"], "-> we", rec["verb"],
      "| because:", rec["downgraded_because"])
# CONDITIONAL FOR -> we suggest | because: ['variable patient values']

CONDITIONAL FOR -> we suggest | because: ['variable patient values']


## Contrast — when all three line up, it's STRONG

Flip the one soft input: make the values **consistent**. Now certainty is solid, the balance is
clear, and values agree — so the rule returns **STRONG FOR** — "we *recommend*." One input is
the entire difference between *suggest* and *recommend*.

In [7]:
rec2 = recommendation_strength("HIGH", "clear_benefit", "consistent")
print(rec2["strength"], rec2["direction"], "-> we", rec2["verb"])   # STRONG FOR -> we recommend

STRONG FOR -> we recommend


## Panel consensus — Delphi voting and the discordance check

A panel must actually *agree*. **Delphi-style voting** runs rounds against a threshold (say
80%); rounds below it trigger discussion and a revote. **Discordance** — the evidence and the
panel disagreeing, or the threshold never met — is flagged and recorded, not averaged away.

In [8]:
def run_delphi(rounds, threshold):
    for i, pct in enumerate(rounds, 1):
        status = "CONSENSUS" if pct >= threshold else "below -> discuss & revote"
        print(f"  Round {i}: {pct}% agree (threshold {threshold}%) -> {status}")
        if pct >= threshold:
            return True
    return False

print("Delphi vote on the CONDITIONAL-FOR wording (threshold 80%):")
reached = run_delphi([66, 88], threshold=80)

evidence_dir, panel_dir = "FOR", "FOR"   # net benefit favors; panel lands conditional-FOR
discordance = (not reached) or (evidence_dir != panel_dir)
print("\nConsensus reached  :", reached)
print("Discordance check  :", "FLAGGED" if discordance else "none (evidence & panel agree)")

Delphi vote on the CONDITIONAL-FOR wording (threshold 80%):
  Round 1: 66% agree (threshold 80%) -> below -> discuss & revote
  Round 2: 88% agree (threshold 80%) -> CONSENSUS

Consensus reached  : True
Discordance check  : none (evidence & panel agree)


## Recap

- The **SoF table** is the scorecard: one row per critical outcome, with **relative** (comparable) and **absolute** (`baseline x RR - baseline`, decision-driving) effects side by side.
- A recommendation = **direction** (for/against) x **strength**; **STRONG** is gated behind certainty **and** a clear balance **and** consistent values — else **CONDITIONAL**.
- You watched a HIGH-certainty benefit produce only a **CONDITIONAL FOR** once values were variable — flip that one input and it becomes **STRONG FOR**.
- Panels agree via **Delphi voting** to a threshold; **discordance** is flagged, not hidden. Judgments stay human; the arithmetic and the strength mapping live in deterministic code.

← Back to the **Summary of Findings & Evidence-to-Decision** article for the full EtD criteria, the real-world screening example, and interview Q&A.